In [2]:
import earthaccess as ea
import xarray as xr
import zarr
from urllib.parse import urlparse
import fsspec
import warnings
import logging

import dask
from dask.distributed import Client, LocalCluster
from coiled import Cluster as CoiledCluster
#import cartopy.crs as ccrs
import numpy as np
import virtualizarr as vz


import obstore
from obstore.auth.earthdata import NasaEarthdataCredentialProvider # Only S3 links access in-region
from obstore.store import HTTPStore, S3Store
from virtualizarr.parsers import DMRPPParser, NetCDF3Parser, HDFParser
from virtualizarr.registry import ObjectStoreRegistry


auth = ea.login()

for _ in (ea, vz, zarr, xr, obstore, dask, fsspec):
    print(f"{_.__name__}: {_.__version__}")

/tmp/ipykernel_61755/3776995716.py:21: DeprecationWarning: Importing ObjectStoreRegistry from VirtualiZarr is deprecated. Please use 'from obspec_utils.registry import ObjectStoreRegistry instead.
  from virtualizarr.registry import ObjectStoreRegistry


Enter your Earthdata Login username:  deanh808
Enter your Earthdata password:  ········


earthaccess: 0.16.0
virtualizarr: 2.4.0
zarr: 3.1.5
xarray: 2025.6.1
obstore: 0.8.2
dask: 2024.5.2
fsspec: 2025.7.0


In [3]:
cloud_opts = {
    "region": "us-west-2",
    "worker_vm_types": ["t3a.medium"],
    "spot_policy": "spot_with_fallback",
    # "arm": False,
    "name": "test-vd",
    "environ": {"EARTHDATA_TOKEN": auth.token["access_token"]}
}

def create_dask_cluster(environment="local",
                        n_workers=4,
                        cloud_opts={}):
    if environment=="local":
        import logging
        print("Creating new local Dask client")
        cluster = LocalCluster(
            n_workers=n_workers,
            threads_per_worker=1,
            silence_logs=logging.ERROR)
    else:
        print("Creating new Coiled Dask client")
        cluster = CoiledCluster(n_workers=n_workers,
                                **cloud_opts)    

    client = Client(cluster)
    return (client, cluster)

def silence_worker_warnings_and_auth(token):
    import warnings
    import logging
    # local instance of earthaccess is explicitely authenticated
    import earthaccess as ea
    import os

    if token:
        os.environ["EARTHDATA_TOKEN"]= token
        auth = ea.login(strategy="environment")
    
    warnings.filterwarnings("ignore")
    for name in ["distributed", "xarray", "py.warnings", "fsspec", "h5netcdf", "h5py"]:
        logging.getLogger(name).setLevel(logging.ERROR)

In [4]:


if "client" not in locals() or "cluster" not in locals():
    client, cluster = create_dask_cluster(
        environment="local",
        n_workers=16,
        cloud_opts=cloud_opts)
client


Creating new local Dask client


/opt/coiled/env/lib/python3.13/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43115 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-xckpa.dask.host/jupyter/proxy/43115/status,
Dashboard: https://cluster-xckpa.dask.host/jupyter/proxy/43115/status,Workers: 16
Total threads: 16,Total memory: 14.55 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43857,Workers: 16
Dashboard: https://cluster-xckpa.dask.host/jupyter/proxy/43115/status,Total threads: 16
Started: Just now,Total memory: 14.55 GiB
Comm: tcp://127.0.0.1:42361,Total threads: 1
Dashboard: https://cluster-xckpa.dask.host/jupyter/proxy/34227/status,Memory: 0.91 GiB
Nanny: tcp://127.0.0.1:33575,


In [5]:
%%capture
client.run(silence_worker_warnings_and_auth, auth.token["access_token"])

In [6]:
results  = ea.search_data(
    short_name="MUR25-JPL-L4-GLOB-v04.2",
    provider="POCLOUD",
    cloud_hosted=True,
    temporal=("2026-01-01", "2026-01-07")
)
len(results)

8

## Using S3 access

In [16]:
# This is dependant on the dataset, for this one we use PODAAC
credentials_endpoint = "https://archive.podaac.earthdata.nasa.gov/s3credentials"
parsed_url = urlparse(results[0].data_links(access="direct")[0])

bucket = parsed_url.netloc

s3_store = S3Store(
    bucket=bucket,
    region="us-west-2",
    credential_provider=NasaEarthdataCredentialProvider(credentials_endpoint),
    virtual_hosted_style_request=False,
    client_options={"allow_http": True},
)
s3_obstore_registry = ObjectStoreRegistry({f"s3://{bucket}": s3_store})

In [17]:
# not doing anything yet
def preprocess_func(ds):
    return ds

xr_combine_nested_kwargs = {
    "concat_dim": "time",  # Concatenate files along the time dimension
    "preprocess": preprocess_func, # Normalize the dataframe 
    "data_vars": "minimal",  # Only load data variables that include the concat_dim
    "coords": "minimal",  # Only load coordinate variables that include the concat_dim
    "compat": "override",  # Avoid coordinate conflicts by picking the first
    "combine_attrs": "override",  # Avoid attribute conflicts by picking the first
}

In [18]:
# not doing anything yet
def preprocess_func(ds):
    return ds

xr_combine_nested_kwargs = {
    "concat_dim": "time",  # Concatenate files along the time dimension
    "preprocess": preprocess_func, # Normalize the dataframe 
    "data_vars": "minimal",  # Only load data variables that include the concat_dim
    "coords": "minimal",  # Only load coordinate variables that include the concat_dim
    "compat": "override",  # Avoid coordinate conflicts by picking the first
    "combine_attrs": "override",  # Avoid attribute conflicts by picking the first
}

In [19]:
# S3 links
granule_data_urls_s3 = [
    granule.data_links(access="direct")[0] for granule in results
]

In [20]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Numcodecs codecs*",
        category=UserWarning,
    )
    group = None
    hdf5_vds = vz.open_virtual_mfdataset(
        urls=granule_data_urls_s3,      # change to s3 links if we want to run it in region
        registry=s3_obstore_registry,   # change to s3_obstore_registry if we are using S3 links (only in-region access)
        parser=HDFParser(),
        parallel="dask",
        combine="nested",
        **xr_combine_nested_kwargs,
    )
hdf5_vds

UnauthenticatedError: The operation lacked valid authentication credentials for path External AWS credential provider: SystemError: <class 'requests.models.Request'> returned a result with an exception set

Debug source:
Unauthenticated {
    path: "External AWS credential provider",
    source: PyErr {
        type: <class 'SystemError'>,
        value: SystemError("<class 'requests.models.Request'> returned a result with an exception set"),
        traceback: Some(
            "Traceback (most recent call last):\n  File \"/opt/coiled/env/lib/python3.13/site-packages/obstore/auth/earthdata.py\", line 182, in __call__\n    credentials = self._refresh_with_basic_auth(self._auth)\n  File \"/opt/coiled/env/lib/python3.13/site-packages/obstore/auth/earthdata.py\", line 204, in _refresh_with_basic_auth\n    with self.session.get(self._credentials_url, allow_redirects=False) as r:\n  File \"/opt/coiled/env/lib/python3.13/site-packages/requests/sessions.py\", line 602, in get\n    return self

## Using HTTP access

In [21]:
access="out_of_region"

parsed_url = urlparse(results[0].data_links(access=access)[0])
domain = parsed_url.netloc

http_store = HTTPStore.from_url(
    f"https://{domain}",
    client_options={
        "default_headers": {
            "Authorization": f"Bearer {ea.__auth__.token['access_token']}",
        },
    },
)
# This is the "base" URI that we need to pass to virtualizarr
https_obstore_registry = ObjectStoreRegistry({f"https://{domain}": http_store})


In [22]:
granule_dmrpp_urls_https = [
    granule.data_links(access="https")[0] + ".dmrpp" for granule in results
]
granule_data_urls_https = [
    granule.data_links(access="https")[0] for granule in results
]
# S3 links
granule_dmrpp_urls_s3 = [
    granule.data_links(access="direct")[0] + ".dmrpp" for granule in results
]
granule_data_urls_s3 = [
    granule.data_links(access="direct")[0] for granule in results
]

In [23]:
# not doing anything yet
def preprocess_func(ds):
    return ds

xr_combine_nested_kwargs = {
    "concat_dim": "time",  # Concatenate files along the time dimension
    "preprocess": preprocess_func, # Normalize the dataframe 
    "data_vars": "minimal",  # Only load data variables that include the concat_dim
    "coords": "minimal",  # Only load coordinate variables that include the concat_dim
    "compat": "override",  # Avoid coordinate conflicts by picking the first
    "combine_attrs": "override",  # Avoid attribute conflicts by picking the first
}

In [24]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="Numcodecs codecs*",
        category=UserWarning,
    )
    group = None
    hdf5_vds = vz.open_virtual_mfdataset(
        urls=granule_data_urls_https,      # change to s3 links if we want to run it in region
        registry=https_obstore_registry,   # change to s3_obstore_registry if we are using S3 links (only in-region access)
        parser=HDFParser(),
        parallel="dask",
        combine="nested",
        **xr_combine_nested_kwargs,
    )
hdf5_vds

<xarray.Dataset> Size: 66MB
Dimensions:           (time: 8, lat: 720, lon: 1440)
Coordinates:
  * time              (time) datetime64[ns] 64B 2026-01-01T09:00:00 ... 2026-...
  * lat               (lat) float32 3kB -89.88 -89.62 -89.38 ... 89.62 89.88
  * lon               (lon) float32 6kB -179.9 -179.6 -179.4 ... 179.6 179.9
Data variables:
    analysed_sst      (time, lat, lon) int16 17MB ManifestArray<shape=(8, 720...
    analysis_error    (time, lat, lon) int16 17MB ManifestArray<shape=(8, 720...
    mask              (time, lat, lon) int8 8MB ManifestArray<shape=(8, 720, ...
    sea_ice_fraction  (time, lat, lon) int8 8MB ManifestArray<shape=(8, 720, ...
    sst_anomaly       (time, lat, lon) int16 17MB ManifestArray<shape=(8, 720...
Attributes: (12/54)
    Conventions:                CF-1.7, ACDD-1.3
    title:                      Daily 0.25-degree MUR SST, Final product
    summary:                    A low-resolution version of the MUR SST analy...
    keywords:                   Oceans > Ocean Temperature > Sea Surface Temp...
    keywords_vocabulary:        NASA Global Change Master Directory (GCMD) Sc...
    standard_name_vocabulary:   NetCDF Climate and Forecast (CF) Metadata Con...
    ...                         ...
    publisher_name:             GHRSST Project Office
    publisher_url:              https://www.ghrsst.org
    publisher_email:            gpc@ghrsst.org
    file_quality_level:         3
    metadata_link:              http://podaac.jpl.nasa.gov/ws/metadata/datase...
    acknowledgment:             Please acknowledge the use of these data with...